# SHAP Explainability — from Bits to Molecules

This notebook shows how to explain NP-Classifier predictions using SHAP:

| Section | Class | Use case |
|---------|-------|----------|
| 1–4 | `NPClassifierSHAP` | Single model — explain one or several class outputs |
| 5–7 | `NPClassifierEnsembleSHAP` | Ensemble — explain all three levels in one figure |

**How it works**  
`shap.GradientExplainer` computes expected gradients for a set of background fingerprints.  
Each of the 6 144 input dimensions encodes a Morgan environment (atom + radius).  
A positive SHAP value means the substructure encoded by that bit *pushed* the predicted probability **up**.

**Prerequisites**  
Trained checkpoints from `01_training.ipynb` — flat model (`checkpoints/flat/`) and hierarchical models (`checkpoints/hierarchical/`).

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, SVG

from torch_np_classifier import (
    NPClassifierFeaturizer,
    NPClassifierLightning,
    NPClassifierEnsemble,
    NPClassifierSHAP,
    NPClassifierEnsembleSHAP,
    fp_index_to_radius_bit,
    decode_predictions,
)

## 1. Load model and prepare data

In [ ]:
FLAT_CKPT = "checkpoints/flat/np_classifier_flat.ckpt"
TRAIN_CSV = "../examples/train_dataset.csv"

model = NPClassifierLightning.load_from_checkpoint(FLAT_CKPT)
model.eval()

df_train = pd.read_csv(TRAIN_CSV)
label_cols = [c for c in df_train.columns if c not in ("key", "SMILES")]
label_names = np.array(label_cols)

featurizer = NPClassifierFeaturizer(radius=2, n_jobs=-1)
print(f"{len(label_names)} label classes loaded.")

## 2. Build the SHAP background

SHAP needs a reference distribution — ~100 training fingerprints is a good balance between stability and speed.

In [ ]:
N_BACKGROUND = 100

rng = np.random.default_rng(42)
bg_smiles = df_train["SMILES"].dropna().tolist()
chosen_idx = rng.choice(
    len(bg_smiles), min(N_BACKGROUND, len(bg_smiles)), replace=False
)
bg_smiles = [bg_smiles[i] for i in chosen_idx]
bg_feat = featurizer.transform(bg_smiles)  # (100, 6144)

print(f"Background fingerprints: {bg_feat.shape}")

## 3. Explain a single prediction

We explain quercetin, restricting the explainer to the top-predicted class for speed.

In [ ]:
import torch
import lightning as L
from torch.utils.data import DataLoader, TensorDataset

MOL_NAME = "quercetin"
SMI = "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12"

feat = featurizer.transform([SMI])  # (1, 6144)

# Get model probability
ds = TensorDataset(torch.tensor(feat, dtype=torch.float32))
loader = DataLoader(ds, batch_size=1, shuffle=False)
t = L.Trainer(enable_progress_bar=False, logger=False, accelerator="auto")
probs = torch.cat(t.predict(model, loader), dim=0).numpy()[0]  # (730,)

top_class = int(np.argmax(probs))
top_class_name = str(label_names[top_class])
print(
    f"Top predicted class: {top_class_name!r}  (idx={top_class}, p={probs[top_class]:.3f})"
)

In [ ]:
# Build SHAP explainer for the top class only
explainer = NPClassifierSHAP(model, bg_feat, class_indices=[top_class])

sv = explainer.explain_smiles(SMI, featurizer)  # (1, 6144, 1)
sv_1d = sv[0, :, 0]  # (6144,)

print(f"SHAP array shape : {sv.shape}")
print(f"Non-zero values  : {np.count_nonzero(sv_1d)}")
print(f"Max / Min SHAP   : {sv_1d.max():.4f} / {sv_1d.min():.4f}")

### 3.1 Top contributing bits

In [ ]:
K = 6
top_idx = explainer.top_feature_indices(sv_1d, k=K, positive_only=True)

print(f"Top-{K} fingerprint bits for '{top_class_name}'\n")
for rank, fp_idx in enumerate(top_idx, 1):
    env_r, bit = fp_index_to_radius_bit(fp_idx, radius=2)
    print(
        f"  #{rank:2d}  fp_idx={fp_idx:5d}  radius={env_r}  morgan_bit={bit:4d}  SHAP={sv_1d[fp_idx]:+.4f}"
    )

### 3.2 Molecule with highlighted bits (SVG)

In [ ]:
svg = explainer.draw_explanation(
    SMI,
    sv_1d,
    featurizer,
    k=K,
    positive_only=True,
    size=(500, 350),
    as_svg=True,
    per_feature_colors=True,
)
display(SVG(svg))

### 3.3 Fragment grid — isolated Morgan environments

In [ ]:
fig = explainer.draw_bit_fragments(
    SMI,
    sv_1d,
    featurizer,
    k=K,
    positive_only=True,
    fragment_size=(160, 160),
    cols=3,
)
plt.show()

### 3.4 Complete explanation figure

Combines the highlighted molecule, a SHAP bar chart, and the fragment grid into a single publication-ready figure.

In [ ]:
fig = explainer.explanation_figure(
    SMI,
    sv_1d,
    featurizer,
    class_name=top_class_name,
    k=K,
    mol_size=(450, 320),
    fragment_size=(160, 160),
)
plt.show()

## 4. Explain multiple predicted classes simultaneously

Build one explainer that covers the top-3 classes — avoids recomputing the background reference.

In [ ]:
N_CLASSES = 3
top_k_classes = np.argsort(probs)[::-1][:N_CLASSES].tolist()

print("Top-3 predicted classes:")
for c in top_k_classes:
    print(f"  idx={c:3d}  p={probs[c]:.3f}  label={label_names[c]!r}")

multi_exp = NPClassifierSHAP(model, bg_feat, class_indices=top_k_classes)
sv_multi = multi_exp.explain_smiles(SMI, featurizer)  # (1, 6144, 3)

In [ ]:
for pos, class_idx in enumerate(top_k_classes):
    cname = str(label_names[class_idx])
    sv_1dc = sv_multi[0, :, pos]
    fig = multi_exp.explanation_figure(
        SMI,
        sv_1dc,
        featurizer,
        class_name=cname,
        k=K,
    )
    plt.show()

## 5. Ensemble SHAP — three-level explanation

`NPClassifierEnsembleSHAP` wraps the ensemble and computes SHAP values for each predicted level (pathway / superclass / class) automatically.

In [ ]:
PATHWAY_CKPT = "checkpoints/hierarchical/pathway/pathway_np_classifier.ckpt"
SUPERCLASS_CKPT = "checkpoints/hierarchical/superclass/superclass_np_classifier.ckpt"
CLASS_CKPT = "checkpoints/hierarchical/class/class_np_classifier.ckpt"

ensemble = NPClassifierEnsemble.from_checkpoints(
    pathway_ckpt=PATHWAY_CKPT,
    superclass_ckpt=SUPERCLASS_CKPT,
    class_ckpt=CLASS_CKPT,
)

ens_shap = NPClassifierEnsembleSHAP(ensemble, bg_feat)
print("Ensemble SHAP explainer ready.")

### 5.1 Inspect raw SHAP data per level

In [ ]:
exp_data = ens_shap.explain_smiles(SMI, featurizer)

print("Voted prediction:")
pred = exp_data["prediction"]
print(f"  pathway    : {pred['pathway']}")
print(f"  superclass : {pred['superclass']}")
print(f"  class      : {pred['class']}")
print(f"  glycoside  : {pred['isglycoside']}")

for level in ("pathway", "superclass", "class"):
    d = exp_data[level]
    if d:
        print(f"\n{level}: labels={d['labels']}, shap={d['shap'].shape}")

### 5.2 Explain a single level

In [ ]:
fig = ens_shap.draw_level(SMI, "superclass", featurizer, k=6)
plt.show()

### 5.3 Combined three-level figure

One panel per level (top voted label only) — pathway → superclass → class.

In [ ]:
fig = ens_shap.explanation_figure(SMI, featurizer, k=6)
plt.show()

## 6. Comparing different molecules

Run the same three-level explanation across multiple molecules to see how structural features differ.

In [ ]:
MOLECULES = {
    "quercetin": "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12",
    "morphine": "CN1CC[C@]23c4c5ccc(O)c4O[C@H]2[C@@H](O)C=C[C@@H]3[C@@H]1C5",
    "caffeine": "Cn1cnc2c1c(=O)n(c(=O)n2C)C",
}

for name, smi in MOLECULES.items():
    print(f"\n{'=' * 60}")
    print(f"  {name.upper()}")
    print("=" * 60)
    fig = ens_shap.explanation_figure(smi, featurizer, k=6)
    fig.suptitle(name.title(), fontsize=13, fontweight="bold", y=1.02)
    plt.show()

## 7. Save figures

All drawing methods return a `matplotlib.Figure` — save with `.savefig()`.

In [ ]:
import os

os.makedirs("explanation_outputs", exist_ok=True)

for name, smi in MOLECULES.items():
    # Full three-level ensemble figure
    fig_ens = ens_shap.explanation_figure(smi, featurizer, k=6)
    path = f"explanation_outputs/{name}_ensemble.png"
    fig_ens.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig_ens)
    print(f"Saved {path}")

    # Individual-level (class) figure
    fig_cls = ens_shap.draw_level(smi, "class", featurizer, k=6)
    path_cls = f"explanation_outputs/{name}_class_level.png"
    fig_cls.savefig(path_cls, dpi=150, bbox_inches="tight")
    plt.close(fig_cls)
    print(f"Saved {path_cls}")